# 01 — Exploration de PaySim

**Objectif :** comprendre les données avant de construire quoi que ce soit (issue #6).

- **Source :** [PaySim](https://www.kaggle.com/datasets/ealaxi/paysim1), transactions mobile money
  **simulées** (aucune donnée réelle). 6 362 620 transactions sur 743 heures (`step`).
- **Méthode :** l'exploration se fait sur un échantillon de 10 % (`random_state=42`) pour aller vite ;
  la règle `isFlaggedFraud` (section 7) et le nettoyage (section 8) utilisent le fichier complet.
- Les sorties de ce notebook sont retirées au commit par nbstripout : il faut l'exécuter pour voir
  les résultats.

## 1. Chargement

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Le notebook est dans ml-service/notebooks/ : on rend importable le package app/.
RACINE = Path.cwd().resolve()
if RACINE.name == "notebooks":
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))

from app.donnees import CHEMIN_PARQUET, charger_paysim, nettoyer  # noqa: E402

# Couleurs validées (contraste et daltonisme) : bleu = transactions normales, orange = fraudes.
BLEU, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e4e0",
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
})
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
df = charger_paysim(frac=0.1, random_state=42)

print(f"{len(df):,} transactions, {int(df['isFraud'].sum()):,} fraudes "
      f"({df['isFraud'].mean():.3%})")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.0f} Mo")
df.head()

In [ ]:
df.dtypes

**Constat.** 636 262 transactions, dont 817 fraudes (0,13 %, environ 1 sur 780) : la classe fraude est très minoritaire. C'est pour cela que l'accuracy est inutile ici : un modèle qui répond toujours « normale » obtiendrait 99,87 % d'accuracy sans détecter une seule fraude. On suivra la PR-AUC, le rappel et la précision sur la classe fraude. Les types de colonnes sont fixés par `charger_paysim()` : l'échantillon tient en ~60 Mo.

## 2. Types de transaction

In [ ]:
repartition = (
    df["type"].value_counts()
    .to_frame("transactions")
    .assign(part=lambda t: t["transactions"] / t["transactions"].sum())
)
repartition.style.format({"transactions": "{:,}", "part": "{:.1%}"})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
donnees = repartition.sort_values("transactions")
barres = ax.barh(donnees.index.astype(str), donnees["transactions"], color=BLEU, height=0.6)
ax.bar_label(barres, labels=[f"{p:.1%}" for p in donnees["part"]], padding=4)
ax.set_title("Nombre de transactions par type (échantillon 10 %)")
ax.set_xlabel("Transactions")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:,.0f}".replace(",", " "))
ax.grid(axis="y", visible=False)
plt.show()

**Constat.** CASH_OUT (35 %) et PAYMENT (34 %) dominent, suivis de CASH_IN (22 %). TRANSFER ne représente que 8 % des transactions et DEBIT moins de 1 %.

## 3. Montants

Les montants vont de quelques unités à plusieurs millions : en échelle normale, presque toutes
les transactions seraient tassées dans la première barre. On utilise donc une **échelle
logarithmique** (chaque graduation = ×10).

In [ ]:
print(f"Montants nuls : {(df['amount'] == 0).sum()}")
df.groupby("type", observed=True)["amount"].describe()

In [ ]:
positifs = df[df["amount"] > 0]  # log(0) n'existe pas
classes = np.logspace(np.log10(positifs["amount"].min()), np.log10(positifs["amount"].max()), 60)

fig, ax = plt.subplots()
ax.hist(positifs["amount"], bins=classes, color=BLEU, edgecolor="white", linewidth=0.5)
ax.set_xscale("log")
ax.set_title("Distribution des montants (échelle logarithmique)")
ax.set_xlabel("Montant")
ax.set_ylabel("Transactions")
plt.show()

In [ ]:
types = repartition.index.astype(str).tolist()
fig, ax = plt.subplots()
ax.boxplot(
    [positifs.loc[positifs["type"] == t, "amount"] for t in types],
    tick_labels=types, whis=(5, 95), showfliers=False,
    medianprops={"color": ORANGE, "linewidth": 2},
)
ax.set_yscale("log")
ax.set_title("Montants par type (boîte : 25e–75e percentile, moustaches : 5e–95e)")
ax.set_ylabel("Montant (échelle log)")
plt.show()

In [ ]:
# Comparaison fraude / normale sur les seuls types où la fraude existe (voir section 4),
# en part de chaque groupe (chaque courbe somme à 100 %) : les deux groupes n'ont pas du tout
# la même taille. density=True serait trompeur ici : avec des classes logarithmiques, il divise
# par la largeur de classe et gonfle artificiellement les petits montants.
cibles = positifs[positifs["type"].isin(["TRANSFER", "CASH_OUT"])]

fig, ax = plt.subplots()
for valeur, nom, couleur in [(0, "Normale", BLEU), (1, "Fraude", ORANGE)]:
    montants = cibles.loc[cibles["isFraud"] == valeur, "amount"]
    ax.hist(montants, bins=classes, weights=np.full(len(montants), 1 / len(montants)),
            histtype="step", linewidth=2, color=couleur, label=nom)
ax.set_xscale("log")
ax.set_title("Montants des fraudes et des transactions normales (TRANSFER et CASH_OUT)")
ax.set_xlabel("Montant (échelle log)")
ax.set_ylabel("Part du groupe")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.legend(frameon=False)
plt.show()

cibles.groupby("isFraud")["amount"].describe()

In [ ]:
# PaySim plafonne les montants à 10 000 000 : est-ce propre aux fraudes ?
plafond = cibles["amount"] == 10_000_000
cibles[plafond].groupby("isFraud").size().rename("transactions à 10 000 000 exactement")

**Constat.** Les montants s'étalent sur plusieurs ordres de grandeur (de quelques centimes à 69 millions), d'où l'échelle logarithmique. Ils dépendent fortement du type : médiane ≈ 3 000 pour DEBIT, ≈ 9 500 pour PAYMENT, ≈ 145 000 pour CASH_IN et CASH_OUT, ≈ 490 000 pour TRANSFER.

Sur TRANSFER et CASH_OUT, les fraudes portent sur des montants plus élevés (médiane ≈ 450 000 contre ≈ 172 000), mais les deux distributions se recouvrent largement : **le montant seul ne suffira pas** à détecter la fraude.

Le simulateur plafonne les montants à 10 000 000, pour les fraudes (20) comme pour les transactions normales (300) : ce plafond n'est pas un signal de fraude. Une seule transaction a un montant nul (une fraude CASH_OUT), à examiner avec les soldes.

## 4. Taux de fraude par type

In [ ]:
par_type = (
    df.groupby("type", observed=True)["isFraud"]
    .agg(transactions="size", fraudes="sum", taux="mean")
    .sort_values("fraudes", ascending=False)
)
par_type.style.format({"transactions": "{:,}", "fraudes": "{:,}", "taux": "{:.3%}"})

In [ ]:
types_fraude = set(par_type.index[par_type["fraudes"] > 0].astype(str))
assert types_fraude == {"TRANSFER", "CASH_OUT"}, f"Types avec fraude inattendus : {types_fraude}"
print("Vérifié : la fraude ne concerne que TRANSFER et CASH_OUT.")

**Constat.** La fraude ne concerne que **TRANSFER** (415 fraudes, 0,78 % des virements) et **CASH_OUT** (402 fraudes, 0,18 % des retraits) ; les trois autres types n'en ont aucune (vérifié par `assert`). C'est le scénario simulé par PaySim : le fraudeur prend le contrôle d'un compte, vire l'argent vers un compte qu'il contrôle (TRANSFER), puis le retire (CASH_OUT).

Piste pour le notebook 02 : n'entraîner le modèle que sur ces deux types. Décision à prendre en 02, pas ici.

## 5. Valeurs manquantes et doublons

In [ ]:
print(f"Valeurs manquantes : {int(df.isna().sum().sum())}")
print(f"Lignes en double   : {int(df.duplicated().sum())}")
df.isna().sum().to_frame("valeurs manquantes").T

In [ ]:
# Un client (nameOrig) a-t-il un historique ? Utile pour savoir si le notebook 02
# pourra calculer des variables « par client » (nombre de transactions, montant habituel...).
par_client = df["nameOrig"].value_counts()
print(f"Clients émetteurs distincts : {par_client.size:,} pour {len(df):,} transactions")
print(f"Transactions par client : max {par_client.max()}, "
      f"clients avec plus d'une transaction : {(par_client > 1).sum():,}")

# Les destinataires « M... » sont des commerçants : PaySim ne suit pas leurs soldes.
commercant = df["nameDest"].str.startswith("M")
df.loc[commercant, "type"].astype(str).value_counts().rename("destinataire commerçant (M...)")

**Constat.** Aucune valeur manquante et aucun doublon : il n'y a rien à imputer ni à supprimer.

Presque chaque client émetteur n'apparaît qu'une fois (636 172 clients pour 636 262 transactions, au plus 2 transactions par client). **Il n'y a donc pas d'historique par client** : le notebook 02 ne pourra pas s'appuyer sur des variables du type « nombre de transactions du client » ou « montant habituel ». Les variables viendront de la transaction elle-même (montant, type, soldes, heure).

Les destinataires commerçants (« M... ») ne concernent que PAYMENT, type sans fraude.

## 6. Incohérences de soldes

Pour une transaction qui **débite** l'émetteur, on s'attend à :

- côté émetteur : `oldbalanceOrg − amount = newbalanceOrig` ;
- côté destinataire : `oldbalanceDest + amount = newbalanceDest`.

On calcule l'écart à ces égalités. On se limite à **TRANSFER et CASH_OUT** : ce sont les seuls types
avec de la fraude, et les égalités ci-dessus n'ont pas de sens pour les autres (CASH_IN crédite
l'émetteur, les commerçants de PAYMENT n'ont pas de solde suivi).

Tolérance : **1 unité monétaire**. En dessous, l'écart vient des arrondis des nombres à virgule
(ex. 186 344,56 − 89 698,27 − 96 646,30 donne −0,0100000001 et non 0).

On **mesure** ces incohérences sans les corriger ni supprimer les lignes : si elles sont plus
fréquentes chez les fraudes, ce sont des signaux, que le notebook 02 transformera en variables.

In [ ]:
TOLERANCE = 1.0

cibles = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
cibles["ecart_orig"] = cibles["oldbalanceOrg"] - cibles["amount"] - cibles["newbalanceOrig"]
cibles["ecart_dest"] = cibles["oldbalanceDest"] + cibles["amount"] - cibles["newbalanceDest"]

# Classement de chaque transaction selon la cause de l'écart, côté émetteur puis destinataire.
cibles["cause_orig"] = np.select(
    [
        cibles["ecart_orig"].abs() <= TOLERANCE,
        (cibles["oldbalanceOrg"] == 0) & (cibles["newbalanceOrig"] == 0),
        (cibles["newbalanceOrig"] == 0) & (cibles["amount"] > cibles["oldbalanceOrg"]),
    ],
    ["cohérent", "soldes à 0 avant et après", "montant > solde, solde ramené à 0"],
    default="autre écart",
)
cibles["cause_dest"] = np.select(
    [
        cibles["ecart_dest"].abs() <= TOLERANCE,
        (cibles["oldbalanceDest"] == 0) & (cibles["newbalanceDest"] == 0),
    ],
    ["cohérent", "soldes à 0 avant et après"],
    default="autre écart",
)

GROUPES = {0: "Normale", 1: "Fraude"}
ORDRE = ["cohérent", "soldes à 0 avant et après", "montant > solde, solde ramené à 0", "autre écart"]
causes_orig = pd.crosstab(cibles["cause_orig"], cibles["isFraud"], normalize="columns")
causes_dest = pd.crosstab(cibles["cause_dest"], cibles["isFraud"], normalize="columns")
causes_orig = causes_orig.reindex([c for c in ORDRE if c in causes_orig.index])
causes_dest = causes_dest.reindex([c for c in ORDRE if c in causes_dest.index])
display(causes_orig.rename(columns=GROUPES).style.format("{:.1%}").set_caption("Côté émetteur"))
display(causes_dest.rename(columns=GROUPES).style.format("{:.1%}").set_caption("Côté destinataire"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharex=True)
for ax, causes, titre in [(axes[0], causes_orig, "Côté émetteur"),
                          (axes[1], causes_dest, "Côté destinataire")]:
    y = np.arange(len(causes))
    for decalage, valeur, couleur in [(-0.2, 0, BLEU), (0.2, 1, ORANGE)]:
        barres = ax.barh(y + decalage, causes[valeur], height=0.38, color=couleur,
                         label=GROUPES[valeur])
        ax.bar_label(barres, labels=[f"{p:.0%}" if p >= 0.01 else "" for p in causes[valeur]],
                     padding=3, fontsize=9)
    ax.set_yticks(y, causes.index)
    ax.invert_yaxis()  # première cause en haut
    ax.set_title(titre)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.set_xlim(0, 1.12)
    ax.grid(axis="y", visible=False)
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle("Cohérence des soldes, TRANSFER et CASH_OUT (part de chaque groupe)",
             fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Deux comportements typiques à mesurer : le compte émetteur vidé en une fois, et un
# destinataire dont les soldes restent à 0 malgré l'argent reçu.
indicateurs = pd.DataFrame({
    "compte émetteur vidé": (cibles["newbalanceOrig"] == 0)
    & ((cibles["amount"] - cibles["oldbalanceOrg"]).abs() <= TOLERANCE),
    "soldes destinataire à 0": (cibles["oldbalanceDest"] == 0)
    & (cibles["newbalanceDest"] == 0),
    "type": cibles["type"].astype(str),
    "isFraud": cibles["isFraud"],
})
(indicateurs.groupby(["type", "isFraud"])[["compte émetteur vidé", "soldes destinataire à 0"]]
 .mean().rename(index=GROUPES, level="isFraud").style.format("{:.1%}"))

In [ ]:
# La transaction à montant nul repérée en section 3.
df.loc[df["amount"] == 0]

**Constat.** Le résultat est l'inverse de l'intuition, et c'est le signal le plus fort trouvé jusqu'ici.

- **Côté émetteur, les fraudes sont cohérentes (99,9 %), les transactions normales ne le sont presque jamais (10 %).** Les normales ont souvent des soldes à 0 avant et après (47 %) ou un montant supérieur au solde, ramené à 0 (43 %) : le simulateur ne suit pas bien leurs soldes. Les fraudes, elles, **vident le compte en une fois** : montant = solde initial, nouveau solde à 0 (99,8 % des CASH_OUT frauduleux, 97,3 % des TRANSFER).
- **Côté destinataire, la moitié des fraudes (50 %) laisse les soldes à 0 avant et après**, contre 0,1 % des normales. Ce sont les virements frauduleux : 99,3 % des TRANSFER frauduleux, aucun CASH_OUT.
- La transaction à montant nul est une fraude CASH_OUT sur un compte déjà vide (solde 0) : un artefact du simulateur, sans impact sur l'analyse.

Conséquence pour le notebook 02 : les écarts de soldes (`ecart_orig`, `ecart_dest`) et ces deux indicateurs sont de très bons candidats de variables. Il ne faut surtout pas « corriger » ces soldes au nettoyage : on effacerait le signal.

## 7. La règle existante `isFlaggedFraud`

PaySim fournit une règle déjà en place : `isFlaggedFraud = 1` signale une transaction comme
suspecte. Que détecte-t-elle, et que rate-t-elle ?

Cette analyse se fait sur le **fichier complet** : la règle signale si peu de transactions
qu'un échantillon de 10 % pourrait n'en contenir aucune.

In [ ]:
complet = charger_paysim()
signalees = complet[complet["isFlaggedFraud"] == 1]
print(f"Transactions signalées par la règle : {len(signalees)} sur {len(complet):,} "
      f"(échantillon : {int(df['isFlaggedFraud'].sum())})")
signalees["type"].astype(str).value_counts()

In [ ]:
# Que signale la règle ? On regarde ce que les transactions signalées ont en commun.
signalees[["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]] \
    .describe().loc[["min", "max"]]

In [ ]:
virements = complet[complet["type"] == "TRANSFER"]
gros = virements[virements["amount"] > 200_000]
print(f"Virements de plus de 200 000 : {len(gros):,}, dont {int(gros['isFraud'].sum()):,} "
      f"fraudes et {int(gros['isFlaggedFraud'].sum())} signalés")

profil = (
    (virements["amount"] > 200_000)
    & (virements["oldbalanceOrg"] == virements["newbalanceOrig"])
    & (virements["oldbalanceDest"] == 0) & (virements["newbalanceDest"] == 0)
)
print(f"Virements > 200 000 au profil des signalés (solde émetteur inchangé, destinataire à 0) : "
      f"{int(profil.sum())}, dont {int(virements.loc[profil, 'isFlaggedFraud'].sum())} signalés")

In [ ]:
from sklearn.metrics import precision_score, recall_score

display(pd.crosstab(complet["isFlaggedFraud"], complet["isFraud"],
                    rownames=["isFlaggedFraud"], colnames=["isFraud"]))

precision = precision_score(complet["isFraud"], complet["isFlaggedFraud"])
rappel = recall_score(complet["isFraud"], complet["isFlaggedFraud"])
print(f"Précision de la règle : {precision:.1%} (quand elle signale, elle a raison)")
print(f"Rappel de la règle    : {rappel:.2%} (part des fraudes qu'elle détecte)")
print(f"Fraudes manquées      : {int(((complet['isFraud'] == 1) & (complet['isFlaggedFraud'] == 0)).sum()):,} "
      f"sur {int(complet['isFraud'].sum()):,}")

**Constat.** La règle `isFlaggedFraud` ne signale que **16 transactions sur 6,36 millions**, toutes des TRANSFER de plus de 350 000.

- **Précision 100 %** : les 16 sont bien des fraudes.
- **Rappel 0,19 %** : elle manque **8 197 fraudes sur 8 213**. Même parmi les virements de plus de 200 000 (2 740 fraudes), elle n'en signale que 16.
- Les 16 ont un profil commun : solde émetteur inchangé (comme si le virement avait été bloqué) et soldes destinataire à 0. Mais 19 virements ont ce profil et seuls 16 sont signalés : la règle exacte ne se reconstruit pas à partir des colonnes disponibles.

C'est la **référence à battre** : une règle fixe sur le montant ne détecte presque rien. L'objectif du sprint (rappel ≥ 90 %) demande un modèle qui combine plusieurs signaux (type, montant, soldes).

## 8. Nettoyage et sauvegarde

Le nettoyage est **minimal** et découle des constats précédents (`app.donnees.nettoyer`, testée) :

- valeurs manquantes refusées (il n'y en a pas) et doublons exacts retirés (il n'y en a pas) ;
- `isFlaggedFraud` retirée : c'est la sortie d'une règle existante, pas une donnée de la transaction,
  et une transaction reçue par l'API ne l'aura pas ;
- **tout le reste est conservé** : les soldes incohérents sont un signal de fraude, et le choix des
  types et des variables se fait au notebook 02.

On l'applique au **fichier complet** (déjà chargé en section 7) : le notebook 03 découpera les données
dans le temps et a besoin de toutes les fraudes.

In [ ]:
propre = nettoyer(complet)
print(f"{len(complet):,} lignes avant, {len(propre):,} après ; colonnes : {list(propre.columns)}")

# Même vérification qu'en section 4, cette fois sur les 6,36 millions de lignes.
types_fraude = set(propre.loc[propre["isFraud"] == 1, "type"].astype(str).unique())
assert types_fraude == {"TRANSFER", "CASH_OUT"}, f"Types avec fraude inattendus : {types_fraude}"
print(f"Vérifié sur le fichier complet : {int(propre['isFraud'].sum()):,} fraudes, "
      "uniquement TRANSFER et CASH_OUT.")

In [ ]:
CHEMIN_PARQUET.parent.mkdir(parents=True, exist_ok=True)
propre.to_parquet(CHEMIN_PARQUET)  # compression snappy par défaut (pyarrow)

relu = pd.read_parquet(CHEMIN_PARQUET)
pd.testing.assert_frame_equal(relu, propre)  # même contenu, mêmes types
print(f"Sauvegardé : {CHEMIN_PARQUET.relative_to(RACINE)} "
      f"({CHEMIN_PARQUET.stat().st_size / 1e6:.0f} Mo, {len(relu):,} lignes) — relu à l'identique.")

del complet, signalees, virements, gros, profil, propre, relu  # libère la mémoire